## Semantic Similarity

In [ ]:
import os
import pandas as pd
from scipy.stats import pearsonr
from similarity_scores import mean_wup

def evaluate_similarity(datasets, methods, pre_process=None, suffix=None):
    def col_name(column, suffix):
        return f"{column}_{suffix}" if suffix else column
    
    os.makedirs("data/results", exist_ok=True)
    
    for dataset in datasets:
        data = pd.read_csv(f"data/{dataset}.csv")
        path = f"data/results/{dataset}.csv"
        results = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
        
        s1, s2 = data["S1"], data["S2"]
        
        if pre_process != None:
            s1 = [pre_process(s) for s in s1]
            s2 = [pre_process(s) for s in s2]

        for _, column, method in methods:
            if column not in results.columns:
                results[col_name(column, suffix)] = method(s1, s2)
            
        results.to_csv(path, index=False)

        print(
            f"{dataset.upper():<12}"
            + " | ".join(
                f"{name}: {pearsonr(results[col_name(c, suffix)], data["score"])[0]:.4}"
                for name, c, _ in methods
            )
        )


datasets = [
    "stss131", "sts12","sts13","sts14","sts15",
    "sts16","sts17","stsb", "sickr", "mrpc"
]

evaluate_similarity(datasets, [("Wu & Palmer", "wup", mean_wup)])

In [13]:
import gensim.downloader as api

w2v_model = api.load("word2vec-google-news-300")

In [25]:
from similarity_scores import embedding_similarity

methods = [
    ("Wu & Palmer", "wup", mean_wup),
    ("Word2Vec", "w2v", lambda s1, s2: embedding_similarity(w2v_model, s1, s2))
]

evaluate_similarity(datasets, methods)

STSS131     Wu & Palmer: 0.3825 | Word2Vec: 0.7587
STS12       Wu & Palmer: 0.0799 | Word2Vec: 0.5937
STS13       Wu & Palmer: 0.2712 | Word2Vec: 0.6517
STS14       Wu & Palmer: 0.2955 | Word2Vec: 0.6001
STS15       Wu & Palmer: 0.3448 | Word2Vec: 0.6308
STS16       Wu & Palmer: 0.1946 | Word2Vec: 0.5383
STS17       Wu & Palmer: 0.1849 | Word2Vec: 0.6807
STSB        Wu & Palmer: 0.2428 | Word2Vec: 0.6274
SICKR       Wu & Palmer: 0.4445 | Word2Vec: 0.7006
MRPC        Wu & Palmer: 0.2065 | Word2Vec: 0.3961


In [ ]:
from sentence_transformers import SentenceTransformer, models

def build_model(model_name):
    transformer = models.Transformer(model_name)
    
    pooling = models.Pooling(
        transformer.get_embedding_dimension(),
        pooling_mode="mean"
    )
    return SentenceTransformer(modules=[transformer, pooling])

ft_model = api.load("fasttext-wiki-news-subwords-300")
albert_model = build_model("albert/albert-base-v2")
roberta_model = build_model("FacebookAI/roberta-base")
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
from similarity_scores import bert_similarity

methods = [
    ("Wu & Palmer", "wup", mean_wup),
    ("Word2Vec", "w2v", lambda s1, s2: embedding_similarity(w2v_model, s1, s2)),
    ("FastText", "ft", lambda s1, s2: embedding_similarity(ft_model, s1, s2)),
    ("ALBERT", "albert", lambda s1, s2: bert_similarity(albert_model, list(s1), list(s2))),
    ("RoBERTa", "roberta", lambda s1, s2: bert_similarity(roberta_model, list(s1), list(s2))),
    ("SBERT", "sbert", lambda s1, s2: bert_similarity(sbert_model, list(s1), list(s2)))
]

evaluate_similarity(datasets, methods)

In [ ]:
from util import lemma_filter, stopword_filter, stop_lemma_filter, alphabet_filter

per_processes = [
    ("lemma", lemma_filter),
    ("stop", stopword_filter), 
    ("stop_lemma", stop_lemma_filter), 
    ("alphabet", alphabet_filter), 
]

for suffix, process in per_processes:
    evaluate_similarity(datasets, methods, process, suffix)

In [ ]:
import numpy as np
from collections import defaultdict

scores = defaultdict(list)

for dataset in datasets:
    data = pd.read_csv(f"data/{dataset}.csv")
    results = pd.read_csv(f"data/results/{dataset}.csv")
    
    for column in results.columns:
        scores[column].append(pearsonr(results[column], data["score"])[0])

means = {key: np.mean(values) for key, values in scores.items()}

variants = {
    "Default": "", "Lemmatization": "_lemma", "Stopword": "_stop",
    "Lemma & stop": "_stop_lemma", "Alphabet only": "_alphabet",
}

table = pd.DataFrame({
    variant: {name: means[column + suffix] for name, column, _ in methods.items()}
    for variant, suffix in variants.items()
})

print(table.round(4).to_string())

             Default  Lemmatization  Stopword  Lemma & stop  Alphabet only
Wu & Palmer   0.2869         0.2702    0.3060        0.2938         0.2930
Word2Vec      0.6123         0.6186    0.6326        0.6364         0.6193
FastText      0.3181         0.3165    0.4202        0.3889         0.3798
SBERT         0.7613         0.7547    0.7316        0.7295         0.7484
RoBERTa       0.8402         0.8259    0.7810        0.7787         0.8081
ALBERT        0.7627         0.7581    0.7322        0.7277         0.7511


In [ ]:
from itertools import islice
from util import get_verb_adjective, get_synonyms

def ranked_synonyms(word, model):
    if synonyms := get_synonyms(word):
        return word[0], sorted(
            synonyms, 
            key=lambda syn: embedding_similarity(model, word[0], syn),
            reverse=True
        )
    return None

def candidates(sample):
    for row in sample.itertuples(index=False):
        p1 = get_verb_adjective(row.S1)
        p2 = get_verb_adjective(row.S2)
        
        if not p1 or not p2: continue
        
        synonyms = [ranked_synonyms(w, w2v_model) for p in (p1, p2) for w in p]
        
        if any(result is None for result in synonyms): 
            continue
        
        yield row, synonyms  

def synonym_candidates(file_name):
    df = pd.read_csv(file_name)
    sample = df[["S1", "S2", "score"]].sample(frac=1)
        
    for row, (v1, a1, v2, a2) in islice(candidates(sample), 4):
        print(f'"{row.S1}","{row.S2}",{row.score}')
        print(f"{v1[0]} - {v1[1]}\n{a1[0]} - {a1[1]}")
        print(f"{v2[0]} - {v2[1]}\n{a2[0]} - {a2[1]}\n")
        

synonym_candidates("data/sts17.csv")

In [ ]:
evaluate_similarity("data/synonyms.csv", "Synonyms")
evaluate_similarity("data/negations.csv", "Negations")

Pearson correlation coefficients
Synonyms    Wu & Palmer: 0.2630 | Word2vec: 0.6451 | FastText: 0.3140 | SBERT: 0.8056 | RoBERTa: 0.8704 | ALBERT: 0.8757
Negations   Wu & Palmer: 0.2186 | Word2vec: 0.5972 | FastText: 0.1651 | SBERT: 0.7675 | RoBERTa: 0.7246 | ALBERT: 0.7207
